# Phase 2: Parametric Baseline Model (Logistic Regression)
Objective: To establish a statistical baseline for predicting early-stage Alzheimer's Disease and evaluate the precision-recall trade-offs of linear boundaries.

In [64]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import plotly.express as px
import plotly.graph_objects as go

## 1. Data Preparation & Splitting

In [2]:
df = pd.read_csv('data/CSV/Alzheimer_final.csv')
df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV
0,011_S_0003,0.717371,1,4.0,0.081752,1.0,0.014743,0.223724
1,022_S_0004,-0.854411,1,0.0,0.836511,1.0,0.014743,0.223724
2,011_S_0005,-0.148249,1,3.0,1.052156,0.0,0.014743,0.223724
3,100_S_0006,0.614863,0,2.0,0.620865,1.0,0.014743,0.223724
4,011_S_0010,-0.125469,0,1.0,0.513043,1.0,0.014743,0.223724


In [48]:
X = df.drop(columns=['ID', 'Target'])
y = df['Target']

In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 2. Model Training (50% Default Threshold)

In [50]:
log_model = LogisticRegression(class_weight='balanced', max_iter=1000)
log_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [51]:
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

## 3. Baseline Evaluation Metrics

In [52]:
print("--- Logistic Regression Baseline Results ---\n")
print(classification_report(y_test, y_pred, target_names=['Healthy (0)', 'Early-Stage (1)']))

--- Logistic Regression Baseline Results ---

                 precision    recall  f1-score   support

    Healthy (0)       0.66      0.59      0.62       560
Early-Stage (1)       0.55      0.63      0.59       454

       accuracy                           0.61      1014
      macro avg       0.61      0.61      0.60      1014
   weighted avg       0.61      0.61      0.61      1014



In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Predicted Diagnosis", y="Actual Diagnosis", color="Patients"),
                   x=['Healthy (0)', 'Early-Stage (1)'],
                   y=['Healthy (0)', 'Early-Stage (1)'],
                   title="Interactive Confusion Matrix")
fig_cm.update_layout(title_x=0.5, width=600, height=600)
fig_cm.show()

In [55]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Build the interactive ROC Curve
fig_roc = px.area(
    x=fpr, y=tpr,
    title=f'Interactive ROC Curve (AUC = {roc_auc:.4f})',
    labels=dict(x='False Positive Rate (1 - Specificity)', y='True Positive Rate (Sensitivity/Recall)'),
    width=700, height=600,
    color_discrete_sequence=['#1f77b4']
)
# Add the diagonal baseline (50% random guess line)
fig_roc.add_shape(type='line', line=dict(dash='dash', color='gray'), x0=0, x1=1, y0=0, y1=1)
fig_roc.update_layout(title_x=0.5)
fig_roc.show()

## 4. Threshold Optimization (40%)

In [59]:
custom_threshold = 0.40

# 3. Create new predictions based on this lower threshold
y_pred_medical = (y_prob >= custom_threshold).astype(int)
print(f"--- Results with {custom_threshold*100}% Threshold ---")
print(classification_report(y_test, y_pred_medical, target_names=['Healthy (0)', 'Early-Stage (1)']))


--- Results with 40.0% Threshold ---
                 precision    recall  f1-score   support

    Healthy (0)       0.75      0.30      0.43       560
Early-Stage (1)       0.50      0.87      0.64       454

       accuracy                           0.56      1014
      macro avg       0.63      0.59      0.54      1014
   weighted avg       0.64      0.56      0.53      1014



In [60]:
cm_40 = confusion_matrix(y_test, y_pred_medical)
fig_cm = px.imshow(cm_40, text_auto=True, color_continuous_scale='blues',
                   labels=dict(x="Predicted Diagnosis", y="Actual Diagnosis", color="Patients"),
                   x=['Healthy (0)', 'Early-Stage (1)'],
                   y=['Healthy (0)', 'Early-Stage (1)'],
                   title="Confusion Matrix at 40% Threshold")

# Add an annotation explaining the insight directly on the graph
fig_cm.update_layout(title_x=0.5, width=650, height=600)
fig_cm.show()

### 5. Graph The Threshold Trade-Off Curve

In [69]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
tradeoff_df = pd.DataFrame({
    'Threshold': thresholds,
    'Precision': precisions[:-1],
    'Recall': recalls[:-1]
})
tradeoff_melted = tradeoff_df.melt(id_vars='Threshold', value_vars=['Precision', 'Recall'], 
                                   var_name='Metric', value_name='Score')

In [70]:
fig_tradeoff = px.line(tradeoff_melted, x='Threshold', y='Score', color='Metric',
                       title="Precision-Recall Trade-Off Analysis",
                       labels={'Score': 'Percentage (0 to 1)'},
                       color_discrete_sequence=['#ff7f0e', '#1f77b4'])

# DRAW THE INSIGHT: Add a strict vertical line at the 40% mark
fig_tradeoff.add_vline(x=0.40, line_width=3, line_dash="dash", line_color="red", 
                       annotation_text="Our 40% Medical Target", annotation_position="top right")

fig_tradeoff.update_layout(title_x=0.5, width=800, height=500)
fig_tradeoff.show()

> Phase 2 Conclusion: While adjusting the Logistic Regression decision threshold to 40% achieved an excellent Recall of 87%, it collapsed our Precision to 50% and overall Accuracy to 55%. The model became hyper-sensitive, resulting in unacceptable False Positive rates for healthy patients. This proves that linear parametric boundaries are insufficient.